# counterfix quickstart

Find why an LLM call failed, and get a fix that was tested to work. This notebook runs
on a free model: get a free Groq key at https://console.groq.com/keys (no card needed).

In [ ]:
%pip install -q "counterfix[openai]"

## 1. No key needed: the live checks

In [ ]:
import counterfix
from counterfix import CallRecord, Message

record = CallRecord(
    messages=[Message(role="user", content="Answer in exactly one word: sky colour?")],
    response_text="The sky is usually blue on a clear day.",
)
print(counterfix.diagnose(record, judge=False).explain())

## 2. Wrap a real client

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

In [ ]:
from openai import OpenAI

client = counterfix.guard(
    OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.environ["GROQ_API_KEY"])
)
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Write a detailed essay on the monsoon."}],
    max_tokens=64,
)
print(response.choices[0].message.content)
print("live checks:", response.diagnosis.summary())

## 3. A fix that was re-run and checked

In [ ]:
result = counterfix.fix(response)
print(result.explain())

## 4. The deeper look: a judge

The same key also gives a judge (a different model family is picked when it can be).

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Why did Einstein win his third Nobel Prize?"}],
)
print(response.choices[0].message.content[:300], "...")
print(counterfix.diagnose(response).explain())

More: https://github.com/KANSADWALA/CounterFix (docs in `docs/`).